In [1]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from codebleu import calc_codebleu
from code_bert_score import score as codebert_score
import re
import pandas as pd

def compute_metrics(preds, refs, lang="javascript"):
    cb = []
    bleu = []
    meteor = []
    rouge1 = []
    rouge2 = []
    rougeL = []

    smoothie = SmoothingFunction().method1
    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    for i in range(len(preds)):
        p = preds[i]
        r = refs[i]

        # tokenizzazione semplice a whitespace (per BLEU/METEOR)
        p_tok = p.split()
        r_tok = r.split()

        # ----- CodeBLEU per esempio -----
        res = calc_codebleu([r], [p], lang)
        cb.append(res["codebleu"])

        # ----- BLEU -----
        bleu_i = sentence_bleu(
            [r_tok],          # lista di riferimenti tokenizzati
            p_tok,            # ipotesi tokenizzata
            smoothing_function=smoothie,
        )
        bleu.append(bleu_i)

        # ----- METEOR -----
        meteor_i = meteor_score([r_tok], p_tok)
        meteor.append(meteor_i)

        # ----- ROUGE (lavora su stringhe intere) -----
        r_scores = rouge.score(r, p)
        rouge1.append(r_scores["rouge1"].fmeasure)
        rouge2.append(r_scores["rouge2"].fmeasure)
        rougeL.append(r_scores["rougeL"].fmeasure)

    # ----- CodeBERTScore F1 (lista) -----
    _, _, F1, _ = codebert_score(preds, refs, lang=lang)
    f1 = [float(x) for x in F1]

    return cb, f1, bleu, meteor, rouge1, rouge2, rougeL

def extract_js_block(text: str) -> str:
    """
    Se il modello risponde con:
    ```javascript
    // codice
    ```
    estraggo solo il codice. Se non trova nulla, ritorno il testo originale ripulito.
    """
    if text is None:
        return ""
    pattern = r"```(?:javascript|js)?\s*(.*?)```"
    m = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
    if not m:
        return text.strip()
    return m.group(1).strip()

df_real_base=pd.read_json('../../data/dataset/applets/applets_real_new.jsonl',lines=True,orient='records')

df_stral_zs=pd.read_json('../../data/test/generations_all_models_base_only_intent_deepseek (1).jsonl', lines=True, orient='records').merge(df_real_base, on=['row_index'], how='left')
#df_llama2_zs=pd.read_json('generations_all_models_base_zs_llama2.jsonl', lines=True,orient='records').merge(df_real_base, on=['row_index'], how='left')



In [2]:
preds_raw=df_stral_zs.generated.tolist()
refs=df_stral_zs.filter_code.tolist()
prompts=df_stral_zs.instruct_zs.tolist()
LANG      = "javascript"

In [5]:
# Pulizia opzionale: estraggo solo il blocco JS
preds = [extract_js_block(x) for x in preds_raw]

# 4) metriche (tutti insieme, come batch unico)
cb, f1, bleu, meteor, r1, r2, rL = compute_metrics(preds, refs, lang=LANG)

quant_label = "none"
variant = "base_only_intent"

# 5) salvataggio JSONL (prompt + ref + output + metriche)
new_rows = []
for i, (prompt, ref, gen, c, f, b, m, rr1, rr2, rrL) in enumerate(
    zip(prompts, refs, preds, cb, f1, bleu, meteor, r1, r2, rL)
):
    new_rows.append([
        'deepseek',     # model
        variant,      # variant
        i,            # row_id
        prompt,
        ref,
        gen,
        c,
        f,
        b,
        m,
        rr1,
        rr2,
        rrL,
    ])
columns = [
    "model", "variant", "row_id",
    "prompt", "reference", "generated",
    "codebleu", "codebert_f1", "bleu", "meteor",
    "rouge1", "rouge2", "rougeL"]

res=pd.DataFrame(new_rows,columns=columns)

In [6]:
res

,model,variant,row_id,prompt,reference,generated,codebleu,codebert_f1,bleu,meteor,rouge1,rouge2,rougeL
0,deepseek,base_only_intent,0,You are an expert JavaScript developer with sp...,var Hour = Meta.currentUserTime.hour()\r\nvar ...,"{\n ""trigger"": {\n ""condition"": {\n ""...",0.265976,0.588009,0.003787,0.063694,0.130841,0.000000,0.093458
1,deepseek,base_only_intent,1,You are an expert JavaScript developer with sp...,var Hour = Meta.currentUserTime.hour()\r\nvar ...,"{\n ""trigger"": {\n ""condition"": {\n ""...",0.264273,0.575242,0.004532,0.049020,0.048193,0.000000,0.048193
2,deepseek,base_only_intent,2,You are an expert JavaScript developer with sp...,var Hour = Meta.currentUserTime.hour()\r\nvar ...,function isValidTime(time) {\n const hours = ...,0.299780,0.750102,0.007691,0.212272,0.302521,0.153846,0.218487
3,deepseek,base_only_intent,3,You are an expert JavaScript developer with sp...,"if (Twitter.newTweetByUser.Text.indexOf(""SNES""...","{\n ""trigger"": {\n ""user"": {\n ""follo...",0.277750,0.666474,0.004273,0.049020,0.290909,0.075472,0.290909
4,deepseek,base_only_intent,4,You are an expert JavaScript developer with sp...,"if(Feed.newFeedItem.EntryContent.indexOf(""@"") ...",// Define the filter function\nfunction filter...,0.325526,0.679807,0.003555,0.086957,0.092308,0.000000,0.061538
...,...,...,...,...,...,...,...,...,...,...,...,...,...
346,deepseek,base_only_intent,346,You are an expert JavaScript developer with sp...,//var timeOfDay = Meta.triggerTime.weekday()\r...,"{\n ""trigger"": {\n ""and"": [\n {\n ...",0.358637,0.622912,0.000713,0.049650,0.060377,0.015209,0.045283
347,deepseek,base_only_intent,347,You are an expert JavaScript developer with sp...,//var timeOfDay = Meta.triggerTime.weekday()\r...,// Define the trigger\nconst trigger = {\n ev...,0.117183,0.612922,0.000743,0.030336,0.035461,0.007143,0.035461
348,deepseek,base_only_intent,348,You are an expert JavaScript developer with sp...,//var timeOfDay = Meta.triggerTime.weekday()\r...,function filter(tweet) {\n // Check if the ...,0.157409,0.668883,0.003781,0.064135,0.139535,0.046823,0.106312
349,deepseek,base_only_intent,349,You are an expert JavaScript developer with sp...,//var timeOfDay = Meta.triggerTime.weekday()\r...,"{\n ""trigger"": {\n ""platforms"": [""twitter""...",0.308805,0.590463,0.000115,0.032862,0.046875,0.000000,0.031250


In [7]:
res.to_json('temp',lines=True,orient='records')

In [15]:
res

,model,variant,row_id,prompt,reference,generated,codebleu,codebert_f1,bleu,meteor,rouge1,rouge2,rougeL
0,codellama,base_only_intent,0,You are an expert JavaScript developer with sp...,var Hour = Meta.currentUserTime.hour()\r\nvar ...,function filter(event) {\n const currentTime =...,0.222608,0.716480,6.262855e-03,0.057078,0.081081,0.000000,0.054054
1,codellama,base_only_intent,1,You are an expert JavaScript developer with sp...,var Hour = Meta.currentUserTime.hour()\r\nvar ...,function filter(advisory) {\n const currentTim...,0.237026,0.721518,6.100645e-03,0.111663,0.106667,0.000000,0.106667
2,codellama,base_only_intent,2,You are an expert JavaScript developer with sp...,var Hour = Meta.currentUserTime.hour()\r\nvar ...,function filter(alert) {\n const currentTime =...,0.269686,0.721968,5.031399e-03,0.065299,0.109890,0.000000,0.109890
3,codellama,base_only_intent,3,You are an expert JavaScript developer with sp...,"if (Twitter.newTweetByUser.Text.indexOf(""SNES""...",function filterCode(tweet) {\n return tweet.te...,0.366767,0.779633,1.520780e-02,0.088757,0.340426,0.088889,0.340426
4,codellama,base_only_intent,4,You are an expert JavaScript developer with sp...,"if(Feed.newFeedItem.EntryContent.indexOf(""@"") ...",function filterMentions(post) {\n return !post...,0.356578,0.663884,2.764665e-03,0.075758,0.055556,0.000000,0.055556
...,...,...,...,...,...,...,...,...,...,...,...,...,...
346,codellama,base_only_intent,346,You are an expert JavaScript developer with sp...,//var timeOfDay = Meta.triggerTime.weekday()\r...,function filter(tweet) {\n return (\n tweet....,0.107135,0.621134,3.184468e-04,0.030795,0.037175,0.007491,0.029740
347,codellama,base_only_intent,347,You are an expert JavaScript developer with sp...,//var timeOfDay = Meta.triggerTime.weekday()\r...,function filter(tweet) {\n return tweet.user.s...,0.099917,0.569249,5.804657e-09,0.008532,0.024490,0.008230,0.024490
348,codellama,base_only_intent,348,You are an expert JavaScript developer with sp...,//var timeOfDay = Meta.triggerTime.weekday()\r...,function filter(tweet) {\n if (tweet.user.id =...,0.113151,0.623219,6.680387e-04,0.038997,0.074074,0.014925,0.044444
349,codellama,base_only_intent,349,You are an expert JavaScript developer with sp...,//var timeOfDay = Meta.triggerTime.weekday()\r...,function filter(tweet) {\n return tweet.user.i...,0.041938,0.574269,5.569228e-08,0.014205,0.024490,0.008230,0.016327
